# MCP 리소스 정의와 접근

**Skilljar Lessons 07-08 대응**

이 노트북에서 다루는 내용:
1. `@mcp.resource()` 데코레이터로 정적 리소스 정의
2. URI 템플릿으로 동적 리소스 정의
3. 클라이언트에서 리소스 목록 조회 및 읽기
4. 건축공학 리소스 예시 (재료 물성치)

In [ ]:
# ── Setup ──────────────────────────────────────────────
import json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Resource Demo Server")

## §1. 리소스란?

| 구분 | Tools | Resources |
|------|-------|----------|
| 목적 | 함수 실행 (계산, API 호출) | 데이터 제공 (파일, DB) |
| 접근 | 함수 이름으로 호출 | **URI로 접근** |
| 제어 | 모델(LLM)이 호출 결정 | 애플리케이션이 접근 결정 |
| 변경 | 상태 변경 가능 | **읽기 전용** |

## §2. 정적 리소스 정의

고정된 URI로 접근하는 리소스입니다. 설정값, 기준 문서 등에 적합합니다.

In [ ]:
# 정적 리소스 1: 앱 설정
@mcp.resource("config://app/settings")
def get_app_settings() -> str:
    """애플리케이션 설정을 반환합니다."""
    return json.dumps({
        "version": "1.0",
        "debug": False,
        "max_connections": 100
    }, indent=2)


# 정적 리소스 2: 콘크리트 물성치
@mcp.resource("data://materials/concrete")
def get_concrete_properties() -> str:
    """콘크리트 재료 물성치를 반환합니다."""
    return json.dumps({
        "fck_values": {
            "C24": {"fck": 24, "Ec": 25742},
            "C27": {"fck": 27, "Ec": 26871},
            "C30": {"fck": 30, "Ec": 27924},
            "C35": {"fck": 35, "Ec": 29388},
            "C40": {"fck": 40, "Ec": 30722}
        },
        "Ec_formula": "8500 * (fck)^(1/3) MPa",
        "unit_weight": "24 kN/m3"
    }, indent=2, ensure_ascii=False)


print("2개 정적 리소스가 등록되었습니다.")

## §3. 동적 리소스 (Resource Templates)

URI에 `{파라미터}`를 포함하여 동적 데이터를 제공합니다.

In [ ]:
# 동적 리소스: 재료별 물성치
@mcp.resource("data://materials/{material_type}")
def get_material_properties(material_type: str) -> str:
    """지정된 재료의 물성치를 반환합니다.

    Args:
        material_type: 재료 종류 (concrete, steel, rebar)
    """
    materials = {
        "steel": json.dumps({
            "type": "structural_steel",
            "grades": {
                "SS275": {"Fy": 275, "Fu": 410},
                "SS355": {"Fy": 355, "Fu": 490},
                "SM490": {"Fy": 315, "Fu": 490}
            },
            "Es": 200000,
            "unit": "MPa"
        }, indent=2),
        "rebar": json.dumps({
            "type": "reinforcing_bar",
            "grades": {
                "SD400": {"fy": 400, "fu": 560},
                "SD500": {"fy": 500, "fu": 620}
            },
            "Es": 200000,
            "unit": "MPa"
        }, indent=2)
    }
    return materials.get(
        material_type,
        f"'{material_type}' 재료 정보를 찾을 수 없습니다."
    )


print("동적 리소스 (템플릿)가 등록되었습니다.")

## §4. 리소스 조회 및 읽기

등록된 리소스를 프로그래밍 방식으로 조회하고 읽습니다.

In [ ]:
import asyncio

async def demo_resources():
    # 리소스 목록 조회
    resources = await mcp.list_resources()
    print("=== 등록된 리소스 ===")
    for r in resources:
        print(f"  URI: {r.uri}")
        print(f"  이름: {r.name}")
        print()

    # 리소스 템플릿 조회
    templates = await mcp.list_resource_templates()
    print("=== 리소스 템플릿 ===")
    for t in templates:
        print(f"  URI 템플릿: {t.uriTemplate}")
        print(f"  이름: {t.name}")
        print()

    # 정적 리소스 읽기
    print("=== 콘크리트 물성치 ===")
    content = await mcp.read_resource("data://materials/concrete")
    print(content)

await demo_resources()

## §5. 서버 파일 업데이트 (리소스 포함)

리소스가 포함된 서버를 `server.py`에 반영합니다.

In [ ]:
server_code = '''
import json
from mcp.server.fastmcp import FastMCP
from datetime import datetime

mcp = FastMCP("Structural Engineering Server")

# ── Tools ──────────────────────────────────────────
@mcp.tool()
def get_current_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """현재 날짜와 시간을 반환합니다."""
    return datetime.now().strftime(format)

@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    """두 숫자를 더합니다."""
    return a + b

@mcp.tool()
def calculate_area(width: float, height: float, unit: str = "m") -> str:
    """직사각형 면적을 계산합니다."""
    area = width * height
    return f"{area:.2f} {unit}\u00b2"

# ── Resources ──────────────────────────────────────
@mcp.resource("data://materials/concrete")
def get_concrete_properties() -> str:
    """콘크리트 재료 물성치를 반환합니다."""
    return json.dumps({
        "C24": {"fck": 24, "Ec": 25742},
        "C27": {"fck": 27, "Ec": 26871},
        "C30": {"fck": 30, "Ec": 27924},
        "C35": {"fck": 35, "Ec": 29388},
        "C40": {"fck": 40, "Ec": 30722}
    }, indent=2)

@mcp.resource("data://materials/{material_type}")
def get_material_properties(material_type: str) -> str:
    """지정된 재료의 물성치를 반환합니다."""
    materials = {
        "steel": json.dumps({"SS275": {"Fy": 275}, "SS355": {"Fy": 355}, "Es": 200000}),
        "rebar": json.dumps({"SD400": {"fy": 400}, "SD500": {"fy": 500}, "Es": 200000})
    }
    return materials.get(material_type, f"\'{material_type}\' not found")

if __name__ == "__main__":
    mcp.run()
'''

with open("server.py", "w") as f:
    f.write(server_code.strip())

print("server.py가 리소스 포함 버전으로 업데이트되었습니다.")

## 핵심 정리

| 리소스 유형 | 데코레이터 | URI 예시 | 용도 |
|------------|-----------|---------|------|
| 정적 | `@mcp.resource("uri")` | `data://materials/concrete` | 고정 데이터 |
| 동적 (템플릿) | `@mcp.resource("uri/{param}")` | `data://materials/{type}` | 파라미터 기반 |

| 클라이언트 API | 용도 |
|---------------|------|
| `list_resources()` | 정적 리소스 목록 |
| `list_resource_templates()` | 동적 리소스 템플릿 목록 |
| `read_resource(uri)` | 리소스 읽기 |